# U4-L3：跟住来宾（闭环与视觉伺服）

**课程目标**

1. 能把L1的水平方向偏差与L2的实测框高停止阈值放入同一循环，使机器人在受控场地中短时跟随单一移动目标。
2. 能解释“远时跟进、近时停止前进、偏差大时先对准、目标不可用时结束”，并根据静态分支及真机结果调整参数。
3. 能在跟随过程中记录IMU和里程计，结束后说明机器人位移的方向、大小与机身朝向，区分直线位移和行走路程。


## 一、来宾移动后，机器人怎样继续跟上

L1让机器人朝向目标，L2在指定间距处测得框高占比：同一目标、完整可见且拍摄条件相近时，框高占比达到这个数值，可作为停止前进的条件。

现在，来宾缓慢向前走，并在标记处短暂停留。机器人不能只按起步时的一段距离走完就结束，而要重新看画面：目标偏向哪里，是否又远了一些，是否已经足够近？本课将方向和远近两个判断连接起来，在一轮有限时间内持续观察。

先用单一目标做静态判断，再由教师看护安排缓慢移动。来宾保持直立、尽量面向相机，不安排遮挡、突然折返或多人穿行。目标已近时机器人不再前进，目标远离后可以重新跟进；本课不自动后退，也没有人物身份跟踪功能。


## 二、从一次走近到连续跟随

### 2.1 每轮都要回答两个问题

| 观察量 | 含义 | 对指令的影响 |
| --- | --- | --- |
| 框中心x − 图像宽度的一半 | 目标在画面左侧还是右侧 | 决定转向角速度；右侧偏差为正，发负角速度向右转 |
| 框高 ÷ 图像高度 | 与L2停止阈值相比是偏远还是已近 | 小于阈值时可跟进；达到阈值后不再前进 |

框高达到阈值不会结束整个L3实验。程序停住前进后仍继续观察，如果来宾再次远离、框高占比减小，就重新判断是否跟进。若方向偏差很大，先原地转向，暂不前进。

### 2.2 闭环与视觉伺服

先给固定速度和时长、执行期间不根据目标变化修改指令，是本任务中的开环做法。反复读取目标位置并改变下一条指令，是闭环做法。用图像中的框位置和大小作为反馈调整运动，称为**视觉伺服**。

本课每次只走一个短步，随后停住、重新取图。跟随会有间歇，不能保证始终保持精确间距。地面、检测波动和更新间隔都会影响实际结果，需要边看输出边核对现场。


### 2.3 准备检测和标定参数

⚠️ **实验安全**：本课会转向和前进，必须有教师看护并清空路线。目标缓慢移动，看护人员留在侧方。先完成静态判断，再分格进入运动模式；不要“全部运行”。出现碰撞风险或异常运动时按既定按钮流程人工中止，不能等待视觉条件或IMU读数保证安全。

📌 **前置**：已完成L1方向调节和L2框高阈值核对；旧动作、轨迹和其他运动任务已结束。保持与标定相同的目标、头部角度和站姿，跟随时不再单独转头，以免改变框高对应关系。

**STEP 01：连接机器人并加载检测模型。**


In [ ]:
import math
import time
from boosteros.robots.booster import BoosterRobot
from boosteros.brain import Detection

robot = BoosterRobot()
detector = Detection(model="person")
print("已连接：", robot.robot_info.model, robot.robot_info.serial_number)
print("当前模式：", robot.get_mode())


**STEP 02：带入L2实测的停止占比。** `STOP_RATIO`表示在本次目标间距处核对过的框高比例，不是像素数，也不是置信度。若在L2改用了瓶子等目标，本课仍需使用同一个目标及匹配模型；换回来宾时先重做标定。

速度、像素范围和短步时长是待调参数。`TURN_ONLY_PX`大于`CENTER_PX`：前者限制何时可以边走边转，后者用于减少画面中心附近的小幅转动。


In [ ]:
STOP_RATIO = float(input("输入L2实测并核对的停止框高占比："))
SETUP_NOTE = input("填写本次目标、标定间距、头部角度和站姿：").strip()
if not math.isfinite(STOP_RATIO) or not 0 < STOP_RATIO < 1 or not SETUP_NOTE:
    raise ValueError("占比需在0与1之间，并保留标定条件说明")

TARGET_CLASS = "person"  # 更换目标时，检测模型、类别和阈值一起修改
FORWARD_SPEED = 0.10    # m/s，待教师结合L2记录确认；小速度也可能不产生有效行走
KP_YAW = 0.001          # rad/s/像素，沿用L1的偏差方向约定
MAX_YAW = 0.20          # rad/s，待真机核对
CENTER_PX = 20          # 认为基本对准的像素范围
TURN_ONLY_PX = 80       # 偏差较大时只转向，不前进；待本机调试
PULSE_SECONDS = 0.20    # 一次短步等待，读状态还会增加实际指令持续时间
READ_LIMIT = 1.0        # 本次取图与检测耗时上限，不能当作完整图像延迟
RUN_SECONDS = 20.0      # 一轮观察窗口，接口阻塞时不能保证到点立即退出
print("停止占比：", STOP_RATIO, "条件：", SETUP_NOTE)


### 2.4 先检查目标读数能不能使用

`observe_target()`返回三个值：水平偏差、框高占比和读数说明。目标数量不是1、框碰到画面边缘或数据无效时，前两项返回`None`，后续程序结束本轮。碰边检查只能发现一部分截断问题；身体被物体挡住、检测框错误等仍需现场观察，不能由这个检查证明目标始终完整。

`choose_velocity()`只计算，不运动。它返回前进速度、转向角速度和本轮决定。先静态检查这些结果，才能运行运动循环。

**STEP 03：定义取图检查与指令计算。**


In [ ]:
def observe_target():
    started = time.monotonic()
    image = robot.get_image(img_type="rgb")
    results = detector.detect(image.to_numpy())
    targets = []
    for result in results:
        if result.class_name == TARGET_CLASS:
            targets.append(result)
    if not targets:
        return None, None, "未检测到目标"
    if len(targets) != 1:
        return None, None, "检测到多个目标"

    box = targets[0].bbox
    values = [box.center_x, box.center_y, box.width, box.height]
    if not all(math.isfinite(value) for value in values):
        return None, None, "检测框包含无效数值"
    if box.width <= 0 or box.height <= 0:
        return None, None, "检测框大小无效"
    left = box.center_x - box.width / 2
    right = box.center_x + box.width / 2
    top = box.center_y - box.height / 2
    bottom = box.center_y + box.height / 2
    if left <= 2 or top <= 2 or right >= image.width - 2 or bottom >= image.height - 2:
        return None, None, "检测框碰到画面边缘，先检查目标是否完整"
    if time.monotonic() - started > READ_LIMIT:
        return None, None, "本次取图与检测过慢"
    error = box.center_x - image.width / 2
    ratio = box.height / image.height
    return error, ratio, "可用"


def choose_velocity(error, ratio):
    yaw_rate = 0.0
    if abs(error) > CENTER_PX:
        yaw_rate = max(-MAX_YAW, min(MAX_YAW, -KP_YAW * error))
    if ratio >= STOP_RATIO:
        return 0.0, yaw_rate, "已近，停止前进"
    if abs(error) > TURN_ONLY_PX:
        return 0.0, yaw_rate, "先原地对准"
    return FORWARD_SPEED, yaw_rate, "跟进"


**STEP 04：机器人保持不动，改变目标位置检查分支。** 分别观察目标在左侧、右侧、较远和达到阈值时的输出；另做一次没有目标的检查。一次只保留一个目标，移动物体不碰机器人。

下面只打印拟发送的指令。记录画面中的目标位置是否与输出一致；若方向相反或目标不完整，先检查模型、头部朝向、标定和偏差符号。


In [ ]:
error, ratio, note = observe_target()
if error is None:
    print("本轮不能运动：", note)
else:
    vx, yaw_rate, decision = choose_velocity(error, ratio)
    print(f"方向偏差={error:+.1f} px，框高占比={ratio:.3f}")
    print(f"决定：{decision}；拟发vx={vx:.2f} m/s，vyaw={yaw_rate:+.3f} rad/s")


## 三、跟随时另外记录机器人怎样移动

相机的偏差和框高决定本课运动；里程计与IMU用于留下这段运动的观察记录。目标移动了多少，不能由机器人的里程计单独得出。

| 记录量 | 本课用途 |
| --- | --- |
| 里程计x、y | 查看机器人在参考平面中的位置变化 |
| 里程计yaw | 查看机身朝向；与位移方向分开记录 |
| IMU z轴角速度 | 观察采样时绕身体竖直方向转动的快慢，单位rad/s |

正常转向也会产生角速度，本课不再用“与静止样本相差0.5”作为结束跟随的理由。IMU角速度不是倾斜角，也不直接表示侧倾或打滑。

**STEP 05：静止读一次，核对字段和单位。** 此格不清零、不运动。


In [ ]:
odom = robot.get_odom()
imu = robot.get_imu()
print("里程计x/y/yaw：", odom.pose_2d, "单位：m、m、rad")
print("三轴角速度：", imu.angular_velocity, "rad/s")
print("三轴线加速度：", imu.linear_acceleration, "m/s²")


## 四、实训：目标移动，机器人跟随

### 4.1 场地和模式

标出起点、目标缓慢移动的短路线及终点停留区。先做正前方远离—停留，再按场地条件做小幅偏左或偏右；不让来宾走向机器人，不做急转弯。保持目标完整可见，路线内不安排其他人员。改变目标或拍摄姿态后，先重新核对L2停止阈值。

**STEP 06：进入准备姿态。**


In [ ]:
robot.set_mode("prepare")
print("当前模式：", robot.get_mode())


> 由人工支撑机器人站起，确认站稳、空间和路线均满足要求，再运行下一格。头部保持标定时的固定角度。

**STEP 07：进入行走模式。**


In [ ]:
robot.set_mode("walk")
print("当前模式：", robot.get_mode())


### 4.2 运行一轮跟随

**STEP 08：现场确认后运行。** 本格在`walk`下清零里程计并保存实际起始读数。清零只改变参考，不使机器人回到起点；实际读数不必恰好为零。

每轮先取图决定指令，短步末尾读取IMU和里程计，再发送零速度。记录里的三路信息是顺序读取的，不保证来自同一时刻。读状态会增加这次指令持续时间，0.2秒不能理解为精确运动时长。最外层`finally`保证本格正常退出或可处理的异常、中断时尝试发送零速度；通信中断或内核被强制终止时仍须现场人工处置。

“已近”只停止前进，仍可小幅原地修正方向；整个循环继续观察。目标不可用时结束本轮，不重新寻找别的目标。


In [ ]:
APPROVED = False  # 每次由教师检查目标、标定、场地与参数后修改
follow_rows = []
start_pose = None
end_pose = None
follow_reason = "本轮未运行"
if APPROVED and robot.get_mode() == "walk" and input("可跟随？输入“跟随”：").strip() == "跟随":
    try:
        robot.set_velocity(0, 0, 0)
        robot.reset_odom()
        start_pose = list(robot.get_odom().pose_2d)
        started = time.monotonic()
        follow_reason = "达到本轮观察时限"
        while time.monotonic() - started < RUN_SECONDS:
            error, ratio, note = observe_target()
            if error is None:
                follow_reason = note
                break
            vx, yaw_rate, decision = choose_velocity(error, ratio)
            try:
                robot.set_velocity(vx, 0, yaw_rate)
                time.sleep(PULSE_SECONDS)
                wz = robot.get_imu().angular_velocity[2]
                x, y, yaw = robot.get_odom().pose_2d
            finally:
                robot.set_velocity(0, 0, 0)
            follow_rows.append([x, y, yaw, error, ratio, wz, decision])
            print(f"{decision}：偏差{error:+.0f}px，占比{ratio:.3f}；x/y={x:+.2f}/{y:+.2f}m，yaw={yaw:+.2f}rad，wz={wz:+.2f}rad/s")
    except (Exception, KeyboardInterrupt):
        follow_reason = "执行异常或中断，请保留报错和本轮记录"
        raise
    finally:
        robot.set_velocity(0, 0, 0)
    print("本轮结束：", follow_reason)
else:
    print("本轮未启动，请检查现场批准、模式和输入")


### 4.3 停稳后查看方位与距离

**STEP 09：教师确认已经停稳后读取终点。** 若本轮没有启动，就不计算。终点读数在这里重新读取，不能把最后一轮运动中的位置自动当作停稳位置。

`dx、dy`是起点到终点的坐标变化，`hypot(dx,dy)`给出两点间的直线位移估计。机器人转弯或绕路时，沿途走过的路程可能更长。`atan2(dy,dx)`给出位移方向；它与机器人最后朝向是两回事。此处角度相对里程计参考x轴，不是相对教室正北。


In [ ]:
def report_motion(start_pose, end_pose, rows):
    dx = end_pose[0] - start_pose[0]
    dy = end_pose[1] - start_pose[1]
    displacement = math.hypot(dx, dy)
    print(f"参考坐标变化：dx={dx:+.3f} m，dy={dy:+.3f} m")
    print(f"起终点直线位移估计：{displacement:.3f} m（不是沿途路程）")
    if displacement >= 0.05:
        bearing = math.degrees(math.atan2(dy, dx))
        print(f"位移方向：相对里程计参考x轴{bearing:+.1f}°，正角朝参考y轴方向")
    else:
        print("位移较小，不解释方向角；0.05 m只是本次显示条件")
    print(f"起始/结束机身朝向：{start_pose[2]:+.3f}/{end_pose[2]:+.3f} rad")
    if rows:
        peak_wz = max(abs(row[5]) for row in rows)
        print(f"已记录{len(rows)}轮；采样中最大|z轴角速度|={peak_wz:.3f} rad/s")


if start_pose is None:
    print("本轮没有起始记录")
else:
    end_pose = list(robot.get_odom().pose_2d)
    report_motion(start_pose, end_pose, follow_rows)
    print("结束原因：", follow_reason)


### 4.4 重复测试与收尾

第一轮先检查“靠近后不再前进”。条件满足后，重新布置路线，再测试“目标远离后重新跟进”和“小幅偏转后调整方向”。每轮恢复标记起点、核对姿态和参数，并重新给予批准。目标丢失、截断、多目标等退出原因必须单独记录，不能写成跟随完成。

**STEP 10：结束运动，回到准备姿态。**


In [ ]:
robot.set_velocity(0, 0, 0)
robot.set_mode("prepare")
print("当前模式：", robot.get_mode())


> 再按已经验证的人工支撑和阻尼流程扶坐，禁止从行走模式直接切阻尼。收尾不代替对线程、动作等旧任务的检查，本课不创建这些任务。

## 五、观察、记录与检查

### 5.1 视觉跟随记录

| 轮次/版本 | 目标及拍摄条件 | 停止占比/速度/短步时长 | 目标怎样移动 | 出现过的跟随决定 | 结束原因 | 最终卷尺间距 |
| --- | --- | --- | --- | --- | --- | --- |
| 1 | | | | | | |
| 2 | | | | | | |

用输出或录像指出：哪次框高达到阈值后前进速度变为0，哪次目标再次远离后恢复跟进；方向偏差较大时是否先转向。最终间距应在目标和机器人都停稳后测量。使用来宾并以1.2米标定时，核对是否处于本组要求的间距；其他目标使用各自的实测间距。

### 5.2 机器人运动记录

| 起始x/y/yaw | 停稳x/y/yaw | dx/dy | 直线位移估计 | 位移方向/参考轴 | 采样最大角速度 | 卷尺或地面标记核对 |
| --- | --- | --- | --- | --- | --- | --- |
| | | | | | | |

说明机器人最终朝向与位移方向是否相同。IMU和里程计提供估计及采样记录，不能证明机器人准确走过某条真实轨迹，不能由最大角速度判定“安全”或“打滑”。

### 5.3 检查点与思考

- 能从代码指出决定前进和转向的图像量，说明为何不再输入起步间距来规划跟随总路程。
- 能说明“达到停止占比”在L2结束走近、在L3暂停前进而继续观察的区别。
- 能区分目标相对机器人的远近、机器人自身位移、机身朝向及IMU角速度。
- 如果目标只转身、弯腰或被遮挡，框高变化还能说明距离变化吗？应怎样处理？
- 框高在阈值两侧来回波动时，输出为何可能反复切换？先怎样检查采样、阈值和速度？
- 为什么从起点走出一条弯线后，`hypot(dx,dy)`不能当成整段行走路程？


## 六、扩展阅读：跟随的适用条件

本课按每轮检测框控制一个当前可见目标，没有识别人物身份。场景中出现第二个人时直接结束；即使始终只有一个检测框，也不能证明它始终属于同一个人。课堂应保持单一目标和清楚的路线。

框高占比是实测条件下的近距离线索，人体侧身、步态摆动或画面截断会破坏原来的对应关系。因此先安排缓慢、直立、无遮挡的演示。真实服务中的人员跟随还需处理身份、遮挡、动态测距和障碍物，本课尚未实现这些能力。

如果参数调整后仍反复启停，先固定目标与姿态、记录同一位置的波动，再决定是否设置分开的停止和重新前进阈值。新增判断后需要重新验证，不能只靠让动作看上去更流畅就认定间距准确。
